# 02 — Rule-Based Baseline

**Project**: From Clinical Jargon to Plain Language — Medical Text Simplification  
**Purpose**: Apply rule-based jargon substitution, sentence splitting, and parenthetical removal to the Phase 1 test set.  
**Output**: `predictions/rule_based.jsonl`, `results/metrics.csv` (rule_based row)  
**Run**: Top-to-bottom on Colab. CPU only — no GPU needed.


## 0. Colab Setup

Mount Drive and load processed data. Run ONCE per session.

In [ ]:
from google.colab import userdata, drive
import os, sys, shutil

# Mount Drive
drive.mount('/drive')

# Clone repo if not already present
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
GITHUB_URL = f'https://{GITHUB_TOKEN}@github.com/IbrahimHanafy2222/NLP-Project.git'
if not os.path.exists('/content/NLP-Project'):
    import subprocess
    result = subprocess.run(['git', 'clone', GITHUB_URL], capture_output=True, text=True, cwd='/content')
    print(result.stdout or result.stderr)

# Set working directory
project_root = '/content/NLP-Project'
os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
print('Working directory:', os.getcwd())

# Load processed data from Drive if not present
if not os.path.exists('data/processed'):
    shutil.copytree('/drive/MyDrive/NLP_Project/processed', 'data/processed')
    print('Loaded data/processed from Drive')
else:
    print('data/processed already present')


## 1. Setup

In [ ]:
import importlib.util as _ilu
if _ilu.find_spec('textstat') is None:
    get_ipython().system('pip install git+https://github.com/feralvam/easse.git sacrebleu datasets==2.18.0 spacy==3.7.4 pandas==2.2.1 textstat scikit-learn==1.4.1.post1')
    get_ipython().system('python -m spacy download en_core_web_sm')
    print('Packages installed. Runtime restarting — re-run from next cell after restart.')
    import os; os.kill(os.getpid(), 9)
else:
    print('Packages already installed. Continuing.')

In [ ]:
import random, json, os, sys, re
import numpy as np
import pandas as pd
from datasets import load_from_disk

# After kernel restart, cwd resets to /content — navigate back to project root
_COLAB_PROJECT = '/content/NLP-Project'
if os.path.exists(_COLAB_PROJECT):
    os.chdir(_COLAB_PROJECT)
project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.rule_based import RuleBasedSimplifier
from src.metrics import compute_sari, compute_bleu, compute_fkgl

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DICT_PATH = os.path.join(project_root, 'data/medical_dict.json')
PROCESSED_DIR = os.path.join(project_root, 'data/processed')
PREDICTIONS_DIR = os.path.join(project_root, 'predictions')
RESULTS_DIR = os.path.join(project_root, 'results')

print('Imports and seeds set')
print('Project root:', project_root)

## 2. Load Dictionary & Simplifier

In [ ]:
simplifier = RuleBasedSimplifier(DICT_PATH, split_threshold=30, min_fragment=5)
print(f'Dictionary loaded: {len(simplifier.dictionary)} entries')
assert len(simplifier.dictionary) >= 500, f'Dictionary too small: {len(simplifier.dictionary)}'
print('Spot checks:')
checks = {'myocardial infarction': 'heart attack', 'hypertension': 'high blood pressure', 'dyspnea': 'shortness of breath'}
for jargon, plain in checks.items():
    if jargon in simplifier.dictionary:
        print(f'  {jargon!r} -> {simplifier.dictionary[jargon]!r}')


## 3. Load Test Data

In [ ]:
splits = load_from_disk(PROCESSED_DIR)
test_df = splits['test'].to_pandas()
print(f'Test split: {len(test_df)} pairs')
assert len(test_df) == 1046, f'Expected 1046 rows, got {len(test_df)}'
print(test_df[['source', 'target']].head(2).to_string())


## 4. Run Simplifier

In [ ]:
predictions = []
substituted_count = 0
split_count = 0
parenthetical_count = 0

for _, row in test_df.iterrows():
    src = row['source']
    after_paren = simplifier.remove_parentheticals(src)
    after_sub = simplifier.substitute_jargon(after_paren)
    final = simplifier.split_long_sentences(after_sub)

    if after_paren != src:
        parenthetical_count += 1
    if after_sub != after_paren:
        substituted_count += 1
    if final != after_sub:
        split_count += 1

    predictions.append({'source': src, 'prediction': final, 'reference': row['target']})

substitution_rate = substituted_count / len(test_df)
print(f'Total sentences:           {len(test_df)}')
print(f'Parentheticals removed:    {parenthetical_count}')
print(f'Jargon substitutions:      {substituted_count}')
print(f'Sentences split:           {split_count}')
print(f'Substitution coverage:     {substituted_count}/{len(test_df)} ({substitution_rate:.1%})')
assert substitution_rate >= 0.20, f'Coverage too low: {substitution_rate:.1%} (need >= 20%)'
print('Coverage assertion passed')


## 5. Save Predictions

In [ ]:
os.makedirs(PREDICTIONS_DIR, exist_ok=True)
out_path = os.path.join(PREDICTIONS_DIR, 'rule_based.jsonl')
with open(out_path, 'w', encoding='utf-8') as f:
    for row in predictions:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

rows = [json.loads(l) for l in open(out_path)]
assert len(rows) == 1046, f'Expected 1046 rows, got {len(rows)}'
assert all('source' in r and 'prediction' in r and 'reference' in r for r in rows)
print(f'Saved {out_path} ({len(rows)} rows)')
print('Sample:', rows[0])


## 6. Compute Metrics

In [ ]:
sources_list = [r['source'] for r in rows]
preds_list = [r['prediction'] for r in rows]
refs_list = [r['reference'] for r in rows]

print('Computing SARI...')
sari = compute_sari(sources_list, preds_list, refs_list)
print('Computing BLEU...')
bleu = compute_bleu(preds_list, refs_list)
print('Computing FKGL...')
fkgl_input = compute_fkgl(sources_list)
fkgl_output = compute_fkgl(preds_list)
fkgl_delta = fkgl_output - fkgl_input

print(f'SARI:        {sari:.4f}')
print(f'BLEU:        {bleu:.4f}')
print(f'FKGL input:  {fkgl_input:.4f}')
print(f'FKGL output: {fkgl_output:.4f}')
print(f'FKGL delta:  {fkgl_delta:.4f}')

assert fkgl_delta < 0, f'FKGL did not decrease: delta={fkgl_delta:.4f} (Constitution Principle III)'
assert 0 <= sari <= 100
assert 0 <= bleu <= 100
print('All metric assertions passed')


## 7. Save Metrics

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)
metrics_path = os.path.join(RESULTS_DIR, 'metrics.csv')

new_row = pd.DataFrame([{
    'system': 'rule_based',
    'sari': round(sari, 4),
    'bleu': round(bleu, 4),
    'fkgl_input': round(fkgl_input, 4),
    'fkgl_output': round(fkgl_output, 4),
    'fkgl_delta': round(fkgl_delta, 4),
}])

if os.path.exists(metrics_path):
    existing = pd.read_csv(metrics_path)
    existing = existing[existing['system'] != 'rule_based']
    combined = pd.concat([existing, new_row], ignore_index=True)
else:
    combined = new_row

combined.to_csv(metrics_path, index=False)
print(f'Saved {metrics_path}')
print(combined.to_string(index=False))


## 8. Validate

In [ ]:
# --- Dictionary validation (US3) ---
d = json.load(open(DICT_PATH))
assert len(d) >= 500, f'Dictionary too small: {len(d)}'
print(f'Dictionary: {len(d)} entries')

# --- metrics.py smoke test (US3) ---
s = compute_sari(['A'], ['A'], ['A'])
b = compute_bleu(['A'], ['A'])
f = compute_fkgl(['A'])
assert isinstance(s, float) and isinstance(b, float) and isinstance(f, float)
print('src/metrics.py: all three functions importable and return float')

# --- Contract compliance (predictions) ---
rows2 = [json.loads(l) for l in open(os.path.join(PREDICTIONS_DIR, 'rule_based.jsonl'))]
assert len(rows2) == 1046
assert all('source' in r and 'prediction' in r and 'reference' in r for r in rows2)
assert all(r['source'] and r['reference'] for r in rows2)
changed = sum(1 for r in rows2 if r['source'] != r['prediction'])
print(f'predictions/rule_based.jsonl: {len(rows2)} rows, {changed} predictions differ from source')

# --- Contract compliance (metrics) ---
df_check = pd.read_csv(os.path.join(RESULTS_DIR, 'metrics.csv'))
assert 'rule_based' in df_check['system'].values
rb = df_check[df_check['system'] == 'rule_based'].iloc[0]
assert rb['fkgl_delta'] < 0
print(f'results/metrics.csv: rule_based row present, fkgl_delta={rb["fkgl_delta"]:.4f} (negative = simpler)')

print()
print('All checks passed. Phase 2 complete.')
